# ADK 에이전트용 AI 모델

이 노트북에서는 모델을 세 가지 방식으로 연결해 봅니다.

1. **Google Gemini** — [Google AI Studio](https://aistudio.google.com/apikey)에서 API 키 발급 후 **모델 ID 문자열**만 넘기면 됩니다 (강의 기본).
2. **OpenAI** — `LiteLlm` 래퍼로 `openai/gpt-4o-mini` 등 지정 ([LiteLLM 문서](https://google.github.io/adk-docs/agents/models/litellm/)).
3. **Ollama (로컬)** — `LiteLlm` + `ollama_chat/...` ([Ollama 문서](https://google.github.io/adk-docs/agents/models/ollama/)).

공통 문서: [AI Models for ADK agents](https://google.github.io/adk-docs/agents/models/)

## 사전 준비

1. `pip install -r requirements.txt` (`litellm` 포함)
2. `.env` 예시  
   - Gemini: `GOOGLE_API_KEY` 또는 `GEMINI_API_KEY`  
   - OpenAI: `OPENAI_API_KEY`  
3. **Windows + LiteLLM** 사용 시 PowerShell에서 `$env:PYTHONUTF8 = "1"` 권장 (인코딩 오류 방지).

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

# python-dotenv: .env 파일의 KEY=값 을 os.environ 에 넣어 줍니다.
load_dotenv(Path(".env").resolve())
load_dotenv(Path("..") / ".env")

# Gemini용 키가 하나라도 있으면 Gemini 실습 가능
has_gemini = bool(
    os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
)
# OpenAI 대시보드에서 발급한 키
has_openai = bool(os.environ.get("OPENAI_API_KEY"))

# 둘 다 없으면 이 노트북의 API 호출 실습을 할 수 없습니다.
assert has_gemini or has_openai, (
    "최소 하나 설정: GOOGLE_API_KEY 또는 GEMINI_API_KEY (Gemini), "
    "또는 OPENAI_API_KEY (OpenAI)"
)
print("Gemini 키:", "있음" if has_gemini else "없음", "/ OpenAI 키:", "있음" if has_openai else "없음")

Gemini 키: 있음 / OpenAI 키: 있음


## 1) Gemini — 모델 ID 문자열 (강의 기본)

`LlmAgent(model="gemini-2.0-flash")` 처럼 문자열만 넘기면 ADK가 Gemini API로 요청합니다.  
모델 목록: [Gemini API 모델](https://ai.google.dev/gemini-api/docs/models/gemini)

In [3]:
import os

from google.adk.agents import LlmAgent

# 사용할 Gemini 모델 식별자 (문자열). 버전은 문서에서 확인하세요.
GEMINI_MODEL = "gemini-2.0-flash"

if not (
    os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
):
    # 키가 없으면 아래 smoke_test에서 None 이라 건너뜀
    gemini_agent = None
    print("Gemini API 키가 없어 gemini_agent 는 건너뜁니다.")
else:
    # LlmAgent: 대화·도구 호출을 담당하는 ADK의 기본 LLM 에이전트
    # name: 로그·이벤트에서 구분용 식별자
    # instruction: 시스템에 가까운 행동 지침(한국어 답변 등)
    gemini_agent = LlmAgent(
        model=GEMINI_MODEL,
        name="gemini_demo_agent",
        instruction="한국어로 짧고 정확하게 답하세요.",
    )
    print("사용 모델:", gemini_agent.model)

사용 모델: gemini-2.0-flash


## 2) OpenAI + LiteLLM

`LiteLlm(model="openai/모델명")` 형식입니다. `OPENAI_API_KEY`가 환경에 있어야 합니다.

- LiteLLM 제공자 목록: [LiteLLM Providers](https://docs.litellm.ai/docs/providers)

In [4]:
import os

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

# LiteLlm: 여러 클라우드/로컬 LLM을 같은 방식으로 쓰게 해 주는 LiteLLM 라이브러리 래퍼
# "openai/모델명" → LiteLLM이 OpenAI HTTP API로 변환해 호출합니다.
if not os.environ.get("OPENAI_API_KEY"):
    openai_agent = None
    print("OPENAI_API_KEY 가 없어 openai_agent 는 건너뜁니다.")
else:
    openai_agent = LlmAgent(
        model=LiteLlm(model="openai/gpt-4o-mini"),
        name="openai_litellm_demo",
        instruction="한국어로 짧고 정확하게 답하세요.",
    )
    print("OpenAI 경로 모델:", openai_agent.model)

OpenAI 경로 모델: model='openai/gpt-4o-mini' llm_client=<google.adk.models.lite_llm.LiteLLMClient object at 0x0000025B48A28E50>


## 3) Ollama (로컬) + LiteLLM

접두사 **`ollama_chat/`** 만 사용합니다. `OLLAMA_API_BASE` 기본값은 `http://localhost:11434` 입니다.

In [ ]:
import os

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

# Ollama 서버 주소 (다른 PC면 IP:포트로 변경)
os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

# ollama_chat/ 은 ADK 문서에서 권장하는 접두사(ollama/ 만 쓰면 이상 동작할 수 있음)
# 미리 터미널에서 ollama pull 로 모델을 받아 두세요.
ollama_agent = LlmAgent(
    model=LiteLlm(model="ollama_chat/gemma3:latest"),
    name="ollama_litellm_demo",
    instruction="한국어로 답하세요.",
)
print("Ollama 에이전트 이름:", ollama_agent.name)

## 4) Runner로 한 턴 호출해 보기

`InMemorySessionService`로 세션을 만든 뒤 `Runner`가 에이전트를 실행합니다.  
아래는 **키가 설정된 에이전트만** 순서대로 스모크 테스트합니다.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# app_name: 이 앱(에이전트 애플리케이션)의 이름. 세션과 짝을 이룹니다.
APP = "model_smoke_test"
USER = "demo_user"
SESSION = "session_001"

# 메모리에만 세션 저장 (서버 끄면 사라짐)
session_service = InMemorySessionService()


async def smoke_test(agent: LlmAgent, label: str) -> None:
    """한 번 질문하고 답을 받은 뒤 세션을 삭제합니다."""
    # (app, user, session) 조합으로 대화 상자 하나를 만듭니다.
    await session_service.create_session(
        app_name=APP, user_id=USER, session_id=SESSION, state={}
    )
    # Runner: 세션을 불러오고 에이전트 파이프라인을 돌립니다.
    runner = Runner(
        agent=agent,
        app_name=APP,
        session_service=session_service,
    )
    # 사용자 메시지는 role=user, 본문은 Part(text=...) 로 만듭니다.
    msg = types.Content(
        role="user",
        parts=[types.Part(text="1+1은? 한 문장으로.")],
    )
    print(f"--- {label} ---")
    # run_async: 처리 과정이 이벤트 스트림으로 나옵니다.
    async for event in runner.run_async(
        user_id=USER, session_id=SESSION, new_message=msg
    ):
        # is_final_response(): 사용자에게 줄 최종 답이 붙은 이벤트인지
        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            if text:
                print(text)
    # 같은 SESSION으로 다시 테스트하려면 이전 세션을 지우는 편이 안전합니다.
    await session_service.delete_session(app_name=APP, user_id=USER, session_id=SESSION)


# 위 셀들을 실행해 gemini_agent / openai_agent 가 정의되어 있어야 합니다.
if gemini_agent is not None:
    await smoke_test(gemini_agent, "Gemini")
if openai_agent is not None:
    await smoke_test(openai_agent, "OpenAI (LiteLLM)")
# Ollama는 PC에 모델이 없으면 오류가 나므로 주석으로만 안내합니다.
# await smoke_test(ollama_agent, "Ollama")